# IndoMiniBERT – Step 4: MLM Bergaya & Classifier Clickbait

**Anggota 4 | Minggu 4–5**

Notebook ini menjalankan seluruh Step 4 sesuai proposal:
1. Fine-tune **Style MLM** (MLM bergaya) dari checkpoint pretraining Step 3
2. Latih **Classifier Clickbait** – dua varian (pretrained encoder vs random init)
3. Evaluasi dan buat laporan ablation (accuracy, macro F1, confusion matrix)

> **Jalankan di Colab dengan runtime GPU T4** (Runtime → Change runtime type → T4 GPU)


## 0. Setup & Clone Repo

In [ ]:
# Cek GPU yang tersedia
import torch
print(f"CUDA tersedia: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
import os

# Ganti dengan URL repo GitHub kalian
REPO_URL = "https://github.com/AI-IF-TelU/Proyek1-GenAI-BERT-Salamanca.git"
REPO_DIR = "/content/Proyek1-GenAI-BERT-Salamanca"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

os.chdir(REPO_DIR)
print(f"Working dir: {os.getcwd()}")

In [ ]:
!pip install -q -r requirements.txt

## 1. Siapkan Dataset dari Kaggle

Dataset CLICK-ID diunduh dari Kaggle. Anda perlu mengunggah `kaggle.json` atau
menghubungkan Google Drive dengan folder dataset yang sudah ada.

In [ ]:
# Pilih salah satu: Upload kaggle.json atau mount Google Drive

# === Opsi A: Upload kaggle.json ===
# from google.colab import files
# uploaded = files.upload()  # Upload kaggle.json
# !mkdir -p ~/.kaggle
# !cp kaggle.json ~/.kaggle/
# !chmod 600 ~/.kaggle/kaggle.json
# !kaggle datasets download -d andikawilliam/clickid --unzip -p click_id_dataset

# === Opsi B: Mount Google Drive (jika dataset sudah ada di Drive) ===
# from google.colab import drive
# drive.mount('/content/drive')
# import os
# os.environ["INDOMINIBERT_DATASET_DIR"] = "/content/drive/MyDrive/click_id_dataset"

# Verifikasi bahwa file splits sudah ada (dari Step 1)
import os
for split in ["train", "val", "test"]:
    path = f"data/splits/titles_{split}.csv"
    exists = os.path.exists(path)
    print(f"{path}: {'✓' if exists else '✗ TIDAK ADA'}")

In [ ]:
# Verifikasi & Upload checkpoint Step 3 (pretrain_mlm.pt)
import os
from pathlib import Path

os.makedirs('checkpoints', exist_ok=True)
os.makedirs('reports', exist_ok=True)

pretrain_ckpt = 'checkpoints/pretrain_mlm.pt'
if os.path.exists(pretrain_ckpt):
    size_mb = os.path.getsize(pretrain_ckpt) / 1e6
    print(f'✓ {pretrain_ckpt} ditemukan ({size_mb:.1f} MB)')
else:
    print('⚠️ Checkpoint Step 3 (checkpoints/pretrain_mlm.pt) belum ada di Colab.')
    print('Silakan klik tombol Choose Files di bawah untuk mengunggah checkpoints/pretrain_mlm.pt dari laptop (~29.5 MB):')
    try:
        from google.colab import files
        uploaded = files.upload()
        for fname in uploaded.keys():
            if fname.endswith('.pt'):
                target = os.path.join('checkpoints', os.path.basename(fname))
                if os.path.exists(fname) and fname != target:
                    os.replace(fname, target)
                print(f'✓ Berhasil menyimpan ke {target} ({os.path.getsize(target)/1e6:.1f} MB)')
    except ImportError:
        print('Bukan environment Google Colab. Pastikan file diletakkan di checkpoints/pretrain_mlm.pt')


## 2. Fine-Tune Style MLM

Melanjutkan training MLM dari checkpoint pretraining pada judul berlabel dengan
token gaya `[CB]` atau `[NCB]` di posisi 1. Token gaya tidak pernah di-mask.

In [ ]:
# Fine-tune Style MLM – 20 epoch sesuai config (style_cfg.epochs = 20)
!python -m src.finetune_style \
    --epochs 20 \
    --batch-size 64 \
    --lr 1e-4 \
    --device cuda

In [ ]:
# Tampilkan laporan metrics style MLM
import json
with open("reports/style_mlm_metrics.json") as f:
    style_report = json.load(f)

print(f"Model          : {style_report['model']}")
print(f"Epochs selesai : {style_report['epochs_completed']}/{style_report['epochs_requested']}")
print(f"Best val loss  : {style_report['best_val_loss']:.4f}")
print(f"Checkpoint     : {style_report['best_checkpoint']}")
print("\nHistory (epoch, train_loss, val_loss):")
for h in style_report['history']:
    print(f"  epoch {h['epoch']:>2}: train={h['train_loss']:.4f}  val={h['val_loss']:.4f}  ppl={h.get('val_perplexity', 'N/A')}")

In [ ]:
# Tampilkan grafik loss
from IPython.display import Image
Image("reports/style_mlm_loss.png")

## 3. Training Classifier Clickbait (Ablation)

Melatih dua varian:
- **random_init** – encoder dari inisialisasi acak
- **pretrained** – encoder dari checkpoint pretraining Step 3 (ablation)

Classifier menggunakan token `[CLS]` tanpa token gaya.

In [ ]:
# Training classifier – 5 epoch sesuai config (classifier_cfg.epochs = 5)
!python -m src.classifier \
    --epochs 5 \
    --batch-size 64 \
    --lr 5e-5 \
    --device cuda

In [ ]:
# Tampilkan tabel perbandingan ablation
import json
with open("reports/style_classifier_ablation.json") as f:
    ablation = json.load(f)

print("=" * 55)
print("  TABEL PERBANDINGAN ABLATION – TEST SPLIT")
print("=" * 55)
print(f"{'Variant':<20} {'Test Acc':>10} {'Macro F1':>10}")
print("-" * 44)
for row in ablation["comparison_table"]:
    print(f"{row['variant']:<20} {row['test_accuracy']:>10.4f} {row['test_f1_macro']:>10.4f}")
print("=" * 55)

In [ ]:
from IPython.display import Image
Image("reports/classifier_loss.png")

## 4. Evaluasi Lengkap Step 4

Memuat semua checkpoint yang sudah disimpan dan menghasilkan laporan evaluasi
lengkap (test split, confusion matrix, contoh fill-mask dari style MLM).

In [ ]:
!python -m src.evaluate_step4 --device cuda

In [ ]:
from IPython.display import Image
Image("reports/step4_confusion_matrix.png")

In [ ]:
import json
with open("reports/step4_evaluation.json") as f:
    eval_report = json.load(f)

print("\n--- CLASSIFIER RANDOM INIT ---")
rm = eval_report["classifier_random_init"]["test_metrics"]
print(f"  Accuracy : {rm['accuracy']:.4f}")
print(f"  Macro F1 : {rm['f1_macro']:.4f}")
print(f"  Confusion matrix (rows=true, cols=pred):")
for row in rm["confusion_matrix"]:
    print(f"    {row}")

print("\n--- CLASSIFIER PRETRAINED ---")
pm = eval_report["classifier_pretrained"]["test_metrics"]
print(f"  Accuracy : {pm['accuracy']:.4f}")
print(f"  Macro F1 : {pm['f1_macro']:.4f}")
print(f"  Confusion matrix:")
for row in pm["confusion_matrix"]:
    print(f"    {row}")

delta = eval_report["delta_f1_pretrained_minus_random"]
print(f"\nΔ F1 (pretrained − random): {delta:+.4f}")
print(f"Interpretasi: {eval_report['interpretation']}")

print("\n--- CONTOH FILL-MASK DARI STYLE MLM ---")
for i, ex in enumerate(eval_report.get("fill_mask_examples", []), 1):
    print(f"[{i}] [{ex['style'].upper()[:2]}] {ex['original_text'][:70]}")
    for masked, predicted in zip(ex["masked_tokens"], ex["predicted_tokens"]):
        match = "✓" if masked == predicted else "✗"
        print(f"     masked='{masked}' → pred='{predicted}' {match}")

## 5. Ringkasan Deliverable Step 4

Verifikasi semua output yang diperlukan sudah ada.

In [ ]:
import os

deliverables = [
    ("checkpoints/style_mlm.pt",               "Generator bergaya (Step 4 - utama)"),
    ("checkpoints/classifier_pretrained.pt",   "Classifier pretrained"),
    ("checkpoints/classifier_random_init.pt",  "Classifier random init"),
    ("reports/style_mlm_metrics.json",         "Laporan loss style MLM"),
    ("reports/style_mlm_loss.png",             "Grafik loss style MLM"),
    ("reports/style_classifier_ablation.json", "Laporan ablation"),
    ("reports/classifier_loss.png",            "Grafik loss classifier"),
    ("reports/step4_evaluation.json",          "Evaluasi lengkap (F1, acc, confusion matrix)"),
    ("reports/step4_confusion_matrix.png",     "Visualisasi confusion matrix"),
]

print("=" * 65)
print("  CHECKLIST DELIVERABLE STEP 4")
print("=" * 65)
all_ok = True
for path, desc in deliverables:
    exists = os.path.exists(path)
    status = "✓" if exists else "✗"
    size = f"{os.path.getsize(path)/1e6:.1f} MB" if exists else "TIDAK ADA"
    if not exists:
        all_ok = False
    print(f"  {status} {path:<45} {size}")
    print(f"      {desc}")
print("=" * 65)
print(f"\nStatus: {'✓ SEMUA DELIVERABLE LENGKAP' if all_ok else '✗ ADA FILE YANG KURANG'}")

## 6. Download Hasil ke Lokal

In [ ]:
# Zip semua output penting dan download ke lokal
!zip -r step4_outputs.zip \
    checkpoints/style_mlm.pt \
    checkpoints/classifier_pretrained.pt \
    checkpoints/classifier_random_init.pt \
    reports/

from google.colab import files
files.download("step4_outputs.zip")

---

## Catatan Penting untuk Log Interaksi LLM

Saat menulis Log Interaksi LLM, catat:
- Prompt yang dipakai untuk menghasilkan atau memperbaiki kode Step 4
- Nilai loss style MLM per epoch (dari `reports/style_mlm_metrics.json`)
- Tabel perbandingan ablation classifier (dari `reports/step4_evaluation.json`)
- Contoh fill-mask yang berhasil dan yang gagal
- Keputusan hyperparameter (lr, epochs, batch_size)

Kriteria selesai Step 4 (dari proposal):
- ✅ Generator bergaya (`style_mlm.pt`) tersimpan
- ✅ Classifier (`classifier_pretrained.pt`, `classifier_random_init.pt`) tersimpan
- ✅ Tabel perbandingan F1 pretrained vs random tersedia
- ✅ Akurasi, macro F1, dan confusion matrix pada split test tercatat
